<a href="https://colab.research.google.com/github/ionikim/epilepsy_pediatrics_EEG/blob/main/src/01_loading/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ts2vg
!pip install mne
!git clone https://github.com/ionikim/epilepsy_pediatrics_EEG
from pathlib import Path
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from ts2vg import HorizontalVG
import scipy.sparse as sp

In [ ]:
RAW_DIR = Path('epilepsy_pediatrics_EEG/data/raw')
print("RAW_DIR exists:", RAW_DIR.exists())

for f in RAW_DIR.iterdir():
   print("-", f.name)

In [ ]:
# Extract seizure start and length from a .seizures annotation file.
def get_seizure_period(annotation_file):
    with open(annotation_file, "rb") as f:
        byte_array = f.read()
    start = int(bin(byte_array[38])[2:] + bin(byte_array[41])[2:], 2)
    length = byte_array[49]
    return start, length

# Load an EDF file, create a DataFrame, and label seizure periods
def process_edf_with_labels(edf_file, seizure_file):
    raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)
    raw.filter(1., 45.) # band pass filtering
    raw.set_eeg_reference('average') # egg_re_reference

    electrode_names = raw.ch_names
    sfreq           = raw.info['sfreq']

    electrode_names = raw.ch_names
    sfreq           = raw.info['sfreq']
    n_samples       = len(raw.times)
    time_marks      = np.arange(n_samples) / sfreq
    electrode_data  = raw.get_data()

    df = pd.DataFrame(electrode_data.T, columns=electrode_names)
    df.index = time_marks
    df.index.name = 'Time (s)'

    # label
    if seizure_file:
        seizure_start, seizure_length = get_seizure_period(seizure_file)
        seizure_end = seizure_start + seizure_length
        df["label"] = df.index.map(
            lambda t: "ictal" if seizure_start <= t <= seizure_end else "interictal"
        )
        has_seizure = True
    else:
        df["label"] = "interictal"
        has_seizure = False

    return df, has_seizure